In [1]:
import sys
import os

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.LoadupSamples import LoadupSamples
from src.predictionModule.FilterSamples import FilterSamples
from src.predictionModule.MachineModels import MachineModels
from src.common.DataFrameTimeOperations import DataFrameTimeOperations as dfta

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import datetime
import optuna
import random
import torch
import copy

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import logging
formatted_date = datetime.datetime.now().strftime("%d%b%y_%H%M").lower()

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter(fmt="%(asctime)s - %(message)s")
handler.setFormatter(formatter)
#if not logger.hasHandlers():
#    logger.addHandler(handler)
#else:
#    logger.handlers[:] = [handler]

#Output File handler
formatted_str = f"notebook-ydea-leavesLGB-{formatted_date}"
file_handler = logging.FileHandler(f"{formatted_str}.log", mode="w")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Usage
logger.setLevel(logging.INFO)
logger.info("This will print to the notebook's output cell")

c:\Users\kimer\Desktop\RandomOdyssey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
params = {
    "idxAfterPrediction": 5,
    'timesteps': 90,
    'target_option': 'last',
    "LoadupSamples_tree_scaling_standard": True,
    "LoadupSamples_time_scaling_stretch": False,
    "LoadupSamples_time_inc_factor": 1,

    "FilterSamples_q_up": 0.6,
    
    "FilterSamples_cat_over2.0": True,
    "FilterSamples_cat_under20.0": True,
    "FilterSamples_cat_posOneYearReturn": False,
    "FilterSamples_cat_posFiveYearReturn": False,
    "FilterSamples_cat_doubleFiveYearReturn": False,

    "LSTM_val_split": 0.1,
}

In [7]:
timegroup = "group_regOHLCV_over5years"
treegroup = "group_finanTo2011"

logger.info(f"Using treegroup: {treegroup}, timegroup: {timegroup}")

eval_date = datetime.date(year=2025, month=7, day=13)
evaldates = [eval_date - datetime.timedelta(days=i) for i in range(1, 6)]
start_train_date = datetime.date(year=2014, month=1, day=1)
split_Date = datetime.date(year=2025, month=7, day=1)
ls = LoadupSamples(
    train_start_date=start_train_date,
    test_dates=evaldates,
    treegroup=treegroup,
    timegroup=timegroup,
    params=params,
)
ls.load_samples(main_path = "../src/featureAlchemy/bin/")
ls.split_dataset(
    start_date=start_train_date,
    last_train_date=split_Date,
    last_test_date=eval_date
)
fs_pre = FilterSamples(
    Xtree_train = ls.train_Xtree, 
    ytree_train = ls.train_ytree, 
    treenames   = ls.featureTreeNames,
    Xtree_test  = ls.test_Xtree,  
    ytree_test  = ls.test_ytree,
    meta_train  = ls.meta_pl_train, 
    meta_test   = ls.meta_pl_test, 
    params      = params
)
mask_train_pre, mask_test_pre = fs_pre.categorical_masks()
ls.apply_masks(mask_train_pre, mask_test_pre)

In [ ]:
Xtree_train = ls.train_Xtree
ytree_train = ls.train_ytree
Xtree_test  = ls.test_Xtree
ytree_test  = ls.test_ytree

Xtime_train = ls.train_Xtime
ytime_train = ls.train_ytime
Xtime_test  = ls.test_Xtime
ytime_test  = ls.test_ytime

treenames   = ls.featureTreeNames
timenames   = ls.featureTimeNames
meta_train  = ls.meta_pl_train
meta_test   = ls.meta_pl_test

dates_tr = meta_train['date'].unique().sort()
dates_te = meta_test['date'].unique().sort()

nS, nT, nF = Xtime_train.shape

logger.info(f"Mean ytree_train: {np.mean(ytree_train)}, std: {np.std(ytree_train)}")
logger.info(f"Time: nS: {nS}, nT: {nT}, nF: {nF}")
logger.info(f"Tree: nS: {Xtree_train.shape[0]}, nF: {Xtree_train.shape[1]}")

In [ ]:
def geometric_mean_safe(arr):
    arr = np.asarray(arr, dtype=float)
    minv = np.min(arr) if arr.size else 0.0
    shift = -minv + 1e-9 if minv <= 0 else 0.0
    return float(np.exp(np.mean(np.log(arr + shift)))) if arr.size else np.nan

def metric(arr):
    """Custom cluster score function."""
    gm = geometric_mean_safe(arr)
    return gm - 1

def make_design(X, t_win, f_idx):
    Xw: np.ndarray = X[:, -(t_win+1):, f_idx].copy()
    Xw_mid = (Xw-0.5)*2.0
    
    mask_bad = np.zeros(Xw.shape[0], dtype=bool)
    if f_idx == 0:
        bound_bad = 0.2
        mask_bad = np.any((Xw_mid <= (-1+bound_bad)) | (Xw_mid >= (1-bound_bad)), axis=1)
    else:
        bound_bad = 0.0001
        Xw_mid = np.clip(Xw_mid, -1+bound_bad, 1-bound_bad)
    
    Xw = np.arctanh(Xw_mid) + 1.0
    prev = Xw[:, :-1]
    curr = Xw[:, 1:]
    eps = 1e-4
    Xw = (curr / (prev + eps)) - 1.0
    Xw = np.clip(Xw, 1e-5, 1e5)
    
    return Xw.reshape(Xw.shape[0], -1), mask_bad

dates_tr_idx = dfta(meta_train, 'date').getNextLowerOrEqualIndices(dates_tr)
# design matrices
Xd_0, mask_bad = make_design(Xtime_train, 89, 0)
Xd_1, _ = make_design(Xtime_train, 89, 1)

Xd_0 = Xd_0[~mask_bad]
Xd_1 = Xd_1[~mask_bad]
ytree_train = ytree_train[~mask_bad]
dates_tr_idx = dfta(meta_train.filter(~pl.Series(mask_bad)), 'date').getNextLowerOrEqualIndices(dates_tr)

assert not any([i == -1 for i in dates_tr_idx])
nS, nT = Xd_1.shape

In [ ]:
def __top_leaf_labels_per_tree(
    model: lgb.Booster,
    X,
    y,
    tree_n_max: int,
    top_n_max: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    For each tree t in [0, tree_idx_max], compute your score per leaf:
        score = (gmean_y - 1.96 * std_y / sqrt(count_y)) - 1.0  if count_y > 2
                1.0 - 1.0 (=0)                                  otherwise
    Then pick the top `top_n_max` labels by score.
    Returns: int32 array of shape (top_n_max, tree_idx_max) with label ids; pads with -1.
    And score values associated with those labels, shape (top_n_max, tree_idx_max); pads with metric(1.0).
    Assumes y > 0 (for geometric mean).
    """
    booster = model
    leaf_mat = booster.predict(X, pred_leaf=True)  # shape: (n_samples, n_trees_total)
    n_trees_total = leaf_mat.shape[1]
    n_trees = min(tree_n_max, n_trees_total)

    y_arr = np.asarray(y, dtype=float)
    out_label = np.full((top_n_max, n_trees), -1, dtype=np.int32)
    out_score = np.full((top_n_max, n_trees), metric(1.0), dtype=float)

    for t in range(n_trees):
        labels_t = leaf_mat[:, t].astype(np.int32)

        df = pl.DataFrame({"label": labels_t, "y": y_arr})
        gb = df.group_by("label").agg([
            pl.count("y").alias("count_y"),
            pl.mean("y").alias("mean_y"),
            #pl.col("y").log().mean().exp().alias("gmean_y"),
            #pl.sum("y").alias("sum_y"),
            #pl.var("y").alias("var_y"),
            pl.std("y").alias("std_y"),
            #pl.max("y").alias("max_y"),
            #pl.min("y").alias("min_y"),
            #pl.median("y").alias("median_y"),
            #pl.quantile("y", 0.1).alias("q10_y"),
            #pl.quantile("y", 0.25).alias("q25_y"),
            #pl.quantile("y", 0.75).alias("q75_y"),
            #pl.quantile("y", 0.9).alias("q90_y"),
        ]).with_columns([
            (pl.col("mean_y") - 1.96 * pl.col("std_y") / pl.col("count_y").sqrt())
                .alias("_tmp")
        ]).with_columns([
            (pl.col("_tmp") - 1.0).alias("score")
        ])

        sorted_gb = gb.sort(["score", "count_y"], descending=[True, True]).select(["label", "score"])
        arr = sorted_gb.to_numpy()  # shape (n_labels, 2)
        k = min(top_n_max, arr.shape[0])
        if k > 0:
            out_label[:k, t] = arr[:k, 0].astype(np.int32)
            out_score[:k, t] = arr[:k, 1].astype(float)

    return out_label, out_score

def _score_once_lgb(
    t_win: int,
    Xd_tr: np.ndarray,
    ytr_tree: np.ndarray,
    Xd_te: np.ndarray,
    yte_tree: np.ndarray,
    mm: MachineModels,
    do_transform: bool = False,
    tree_n_max: int = 1,
    top_n_max: int = 1,
    min_n_tar: int = 0,
) -> float:

    if do_transform:
        scaler = StandardScaler().fit(Xd_tr)
        Xd_tr = scaler.transform(Xd_tr)
        Xd_te = scaler.transform(Xd_te)
    try:
        logger.disabled = True
        model_lgb, info = mm.run_LGB(
            X_train=Xd_tr,
            y_train=ytr_tree,
            X_test=Xd_te,
            y_test=yte_tree,
        )
    except Exception as e:
        logger.disabled = False
        logger.warning(f"  LGB failed: {e}")
        return metric(1.0)
    finally:
        logger.disabled = False

    #bi = info.get("best_iteration", 1)
    tree_n_max = min(model_lgb.num_trees(), tree_n_max)
    labels_top, scores_top = __top_leaf_labels_per_tree(
        model_lgb, Xd_tr, ytr_tree, tree_n_max=tree_n_max, top_n_max=max(1, top_n_max or 1)
    )
    
    # Top labels by score
    n_trees = len(scores_top[0,:])
    leaf_te = model_lgb.predict(Xd_te, pred_leaf=True)[:, :n_trees]
    
    # If everything is -1 across all ranks, bail out
    if labels_top.size == 0 or np.all(labels_top == -1):
        logger.warning("  LGB failed to generate predictions.")
        return metric(1.0)
    
    # Rank every (rank, tree) pair by descending score
    r_idx, t_idx = np.unravel_index(np.argsort(scores_top.ravel())[::-1], scores_top.shape)
    
    sel_pairs = []  # (tree_idx, rank_idx)
    mask_sel = np.zeros(leaf_te.shape[0], dtype=bool)
    for i in range(len(r_idx)):
        r = r_idx[i]
        t = t_idx[i]
        lbl = labels_top[r, t]
        if lbl == -1:
            continue
        mask_sel |= (leaf_te[:, t] == lbl)
        sel_pairs.append((lbl, int(t), int(r)))
        if mask_sel.sum() >= min_n_tar:
            break
    y_selected = yte_tree[mask_sel]

    logger.info(
        f"  LGB selected {y_selected.size} out of {yte_tree.size} samples "
        f"using {len(sel_pairs)} (lbl, tree,rank) pairs: {sel_pairs}"
    )

    if y_selected.size == 0:
        logger.warning("  LGB failed to select testing values.")
        return metric(1.0)

    score = metric(y_selected)
    logger.info(
        f"LGB t_win={t_win} => "
        f"score={score:.6f}, selected={y_selected.size}/{yte_tree.size}"
    )

    return float(score) if np.isfinite(score) else metric(1.0)

In [ ]:
# ---- knobs (use existing globals if present) ----
device = "cuda" if torch.cuda.is_available() else "cpu"
n_splits = 50
n_test_days = 20

max_training_days = 1301
N = len(dates_tr_idx)
lo = max_training_days - 1                      # min pivot (last train index)
hi = N - n_test_days - 2                      # max pivot
eligible = list(range(lo, hi + 1))
assert len(eligible) >= n_splits, f"Too few eligible pivots ({len(eligible)}) for n_splits={n_splits}"
pivots = sorted(random.sample(eligible, n_splits))

for p in pivots:
    logger.info(f"  Pivot {p}: Date {dates_tr[p]}")

In [ ]:
def make_objective():
    def objective(trial: optuna.Trial) -> float:
        opt_params = copy.deepcopy(params)
        opt_params["LGB_num_boost_round"]           = 10 #trial.suggest_int("LGB_num_boost_round", 40, 60, step=1)
        opt_params["LGB_lambda_l1"]                 = 5e-1 #trial.suggest_float("LGB_lambda_l1", 5e-3, 1e-1, log=True)
        opt_params["LGB_lambda_l2"]                 = 5e-1 #trial.suggest_float("LGB_lambda_l2", 1e-5, 1e-3, log=True)
        opt_params["LGB_feature_fraction"]          = 0.005
        opt_params["LGB_num_leaves"]                = trial.suggest_int("LGB_num_leaves", 50, 550, step=25)
        opt_params["LGB_max_depth"]                 = trial.suggest_int("LGB_max_depth", 3, 15, step=1)
        opt_params["LGB_learning_rate"]             = trial.suggest_float("LGB_learning_rate", 1e-4, 2e-0, log=True)
        opt_params["LGB_min_data_in_leaf"]          = trial.suggest_int("LGB_min_data_in_leaf", 30, 950, step=10)
        opt_params["LGB_min_gain_to_split"]         = trial.suggest_float("LGB_min_gain_to_split", 1e-5, 5e-0, log=True)
        opt_params["LGB_path_smooth"]               = 0.6 #trial.suggest_float("LGB_path_smooth", 1e-2, 5e-1, log=True)
        opt_params["LGB_min_sum_hessian_in_leaf"]   = trial.suggest_float("LGB_min_sum_hessian_in_leaf", 5e-3, 1e-0, log=True)
        opt_params["LGB_max_bin"]                   = trial.suggest_int("LGB_max_bin", 25, 605, step=10)
        opt_params["LGB_early_stopping_rounds"]     = 20

        t_win =             opt_params["t_win"]             = trial.suggest_int("t_win", 4, 35)
        n_training_days =   opt_params["n_training_days"]   = trial.suggest_int("n_training_days", 400, 900, step=100)
        do_transform =      opt_params["do_transform"]      = False
        min_n_tar =         opt_params["min_n_tar"]         = 5
        top_n_max =         opt_params["top_n_max"]         = 5 #trial.suggest_int("top_n_max", 3, 7)  

        logger.info(f"Trial {trial.number} with params: {opt_params}")

        scores = []
        mm = MachineModels(params=opt_params)
        for i in range(n_splits):
            p = pivots[i]

            tr_l_idx = dates_tr_idx[p - n_training_days + 1]
            tr_u_idx = dates_tr_idx[p + 1] - 1
            te_l_idx = dates_tr_idx[p + 1]
            te_u_idx = dates_tr_idx[p + n_test_days + 1] - 1

            # example: get date bounds if needed
            s_tr = slice(tr_l_idx, tr_u_idx)
            s_te = slice(te_l_idx, te_u_idx)
            Xd_tr = np.hstack((Xd_0[s_tr][:, -(t_win):].copy(),Xd_1[s_tr][:, -(t_win):].copy()))
            #Xd_tr = Xtree_train[s_tr].copy()
            ytr_tree = ytree_train[s_tr].copy()
            
            Xd_te = np.hstack((Xd_0[s_te][:, -(t_win):].copy(),Xd_1[s_te][:, -(t_win):].copy()))
            #Xd_te = Xtree_train[s_te].copy()
            yte_tree = ytree_train[s_te].copy()

            #try:
            sc = _score_once_lgb(t_win, Xd_tr, ytr_tree, Xd_te, yte_tree, mm,
                    do_transform=do_transform, 
                    tree_n_max=opt_params["LGB_num_boost_round"]-1, top_n_max=top_n_max, min_n_tar=min_n_tar)
            #except Exception as e:
            #    logger.info(f"Exception during scoring: {e}")
            #    trial.should_prune()
            #    sc = metric(1.0)
            scores.append(sc)

        vals = [v for v in scores if np.isfinite(v)]
        logger.info(f"Scores per splits: {vals}")
        vals = np.array(vals)
        vals_log = np.log(1.0 + vals)
        if len(vals) < (len(scores)//2):
            return 0.0
        return float(np.mean(vals_log)) if len(vals_log) else -np.inf
    return objective

In [ ]:
studytime = 60*60*5
n_startup_trials = 5
n_jobs = 1
studyname = f"optuna_clustering_idea_{formatted_str}"
logger.info(f"Starting Optuna study: {studyname}")

In [ ]:
optuna.logging.enable_propagation()
objective = make_objective()
sampler = optuna.samplers.TPESampler(n_startup_trials=n_startup_trials, n_ei_candidates=3, consider_prior=True, prior_weight=2.0)
study = optuna.create_study(
    study_name=studyname,
    storage="sqlite:///sandbox_optuna.db",
    direction="maximize",
    load_if_exists=True,
    sampler=sampler,
)
#trial = study.ask()          # sampler proposes params
## set a breakpoint here if you like
#value = objective(trial)     # your breakpoints inside objective will stop here
#study.tell(trial, value)     # persist result to the study

In [ ]:
study.optimize(make_objective(), timeout=studytime, n_jobs=n_jobs)

logger.info(f"Best parameters: {study.best_params}")
logger.info(f"Best score: {study.best_value}")

df: pd.DataFrame = study.trials_dataframe()
logger.info("\nTrials DataFrame:")
logger.info(df.sort_values("value").to_string())

param_importances = optuna.importance.get_param_importances(study)
logger.info("Parameter Importances:")
for key, value in param_importances.items():
    logger.info(f"{key}: {value}")